### **Tema**: Anime-style Turn-Based RPG Mobile Game (Honkai: Star Rail)

---
Di UTS kali ini, aku memilih tema **anime-style turn-based RPG mobile game**, game yang kupilih adalah **Honkai: Star Rail** yang tentunya sesuai soal ada di **Google Play Store**

---
### Alasan pemilihan:
- **Honkai: Star Rail** adalah game **RPG berbasis turn-based** dengan gaya visual **anime** yang sangat populer di platform mobile, sesuai dengan tema yang dipilih.
- Game ini memiliki **jumlah review yang sangat banyak**, memudahkan aku untuk mengambil minimal **1.000 review** sesuai dengan perintah soal dengan berbagai variasi rating (1–5 bintang).
- Aku merupakan pemain aktif **Honkai: Star Rail**, jadi konteks isi review **lebih mudah dipahami**, apalagi saat mencoba interpretasi pola rating dan opini pemain nantinya

### Library yang digunakan untuk scrapping data

In [1]:
from google_play_scraper import Sort, reviews
from langdetect import detect, DetectorFactory
import pandas as pd

#### Supaya hasil deteksi bahasa konsisten

In [2]:
DetectorFactory.seed = 0

#### Scrapping

In [3]:
app_id = "com.HoYoverse.hkrpgoversea"  # App ID Honkai: Star Rail (global)
all_reviews = [] 
continuation_token = None # google play tidak ngasih langsung semua review tapi per halaman/batch

In [4]:
# Scraping review secara bertahap sampai > 1400 review mentah
while len(all_reviews) < 1400:
    batch, continuation_token = reviews(
        app_id,
        lang='en',         # target lokal bahasa Inggris
        country='us',      # region US
        sort=Sort.NEWEST,  # review terbaru
        count=200,         # 200 review per batch
        continuation_token=continuation_token
    )
    all_reviews.extend(batch)
    if continuation_token is None:
        break

print("Jumlah review mentah:", len(all_reviews))

# Ubah ke DataFrame
df = pd.DataFrame(all_reviews)

Jumlah review mentah: 1400


Disini data mentahnya aku buat **1400 dulu**, supaya saat difilter *english only* tetap aman diatas **1000 review**

In [5]:
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,98688904-f0e1-4284-a1db-37cb0d2651fc,HuTao,https://play-lh.googleusercontent.com/a-/ALV-U...,"I love this game, it's really F2P friendly, an...",5,0,3.7.0,2025-11-07 19:52:22,None,NaT,3.7.0
1,fa4fd3f9-367a-4915-8e3c-a3376eef0e1d,Emir Karon,https://play-lh.googleusercontent.com/a/ACg8oc...,Child Phainon does not look like Phainon,2,0,None,2025-11-07 19:21:22,None,NaT,None
2,9d060cc8-2e3a-4146-8eda-cc4a79ba0fcb,Kattayan Roy,https://play-lh.googleusercontent.com/a-/ALV-U...,Trash 🗑️🚮 you guys don't care about stelle mai...,1,0,3.7.0,2025-11-07 19:20:35,None,NaT,3.7.0
3,5fd9ef77-5522-48ee-b478-1c12ee64bbf2,Felicity Jade Ebon,https://play-lh.googleusercontent.com/a-/ALV-U...,maybe if this game had a better writer.....,1,1,2.6.0,2025-11-07 19:07:17,None,NaT,2.6.0
4,d9c9f65d-b17c-4391-9b79-8f1658a53a60,Aamir Ashraf,https://play-lh.googleusercontent.com/a/ACg8oc...,50/50 is a complete lie its more like 75/25 on...,1,10,3.7.0,2025-11-07 18:32:27,None,NaT,3.7.0


Secara default, hasil scrappingan dari **google_play_scrapper** seperti ini, namun hanya beberapa kolom yang akan dipakai untuk selanjutnya seperti **review dari user**, inilah yang jadi teks review utama yang akan jadi **input model ML** , **scorenya** sebagai **label/target** yang akan diprediksi, **kapan dia review**, walaupun kemungkinan tidak akan dipakai di model, tapi akan aku pakai untuk EDA nantinya.

---
Sedangkan untuk **reviewID** hanya ID untuk tiap review, jadi aku tidak pakai, **userName** hanya identitas user, tidak ada hubungannya dengan isi **opini/rating**, **userImage** juga **kurang relevan**, malah berpotensi meganggu privasi, **replyContent** adalah balasan dari **developer**, bukan dalam cakupan yang hanya **analisis review pemain**, **appVersion** hanya informasi aplikasi saat review dibuat, bukan cakupan kita, mungkin untuk **tugas riset spesifik** seperti **dampak update tertentu** bisa dipakai, dan terakhir **thumbsUpCount** tidak dipakai karena tidak berkontribusi langsung terhadap pemodelan rating berbasis teks dan berpotensi **menambah komplexitas** tanpa manfaat yang signifikan

#### Pilih dan rename kolom penting

In [6]:
df = df[["content", "score", "at"]].rename(columns={
    "content": "review_text",
    "score": "rating",
    "at": "review_date"
})

Disini, aku pilih dan rename kolom-kolom penting, Kolom *content* aku ubah jadi **review_text** agar jelas ini text review, *score* menjadi **rating** supaya lebih obvious,  dan *at* menjadi **review_date**, supaya jelas itu adalah tanggal review dibuat, agar nama menjadi lebih pendek dan familiar.

#### Buang baris yang tidak punya review

In [7]:
df = df.dropna(subset=["review_text"])

Disini, aku drop baris yang tidak ada reviewnya, karena tidak dapat digunakan sebagai input untuk pemodelan berbasis teks.

#### Fungsi deteksi bahasa inggris

In [8]:
def detect_lang(text):
    try:
        text = str(text).strip() 
        if text == "":
            return "unknown" # mencegah error dengan jika tidak ada isi yang bisa dideteksi bahasanya, langsung labeli uknown
        return detect(text) # mengembalikan kode bahasa, ex eng -> english, id -> indonesia
    except:
        return "error" 

#### Tambahkan kolom bahasa dan filter hanya English

In [9]:
# Deteksi bahasa untuk setiap teks review, hasilnya disimpan di kolom baru 'lang'
df["lang"] = df["review_text"].apply(detect_lang)

# Lihat distribusi bahasa (berapa banyak 'en', 'id', 'unknown', 'error', dll) sebelum difilter
print("Distribusi bahasa sebelum filter:")
print(df["lang"].value_counts())

# Ambil hanya review yang terdeteksi sebagai bahasa Inggris ('en') dan buat DataFrame baru
df_en = df[df["lang"] == "en"].copy()

# Cek berapa banyak review yang tersisa setelah disaring jadi bahasa Inggris saja
print("Jumlah review bahasa Inggris:", len(df_en))

Distribusi bahasa sebelum filter:
lang
en       1068
cy         74
id         28
vi         23
so         18
af         17
da         17
ca         15
it         14
fr         13
no         12
pl         10
es         10
de          9
et          9
sl          6
ru          5
sq          5
zh-cn       5
tl          5
nl          4
ro          4
error       4
sw          3
fi          3
ko          3
sk          2
pt          2
ar          2
th          2
hr          2
ja          1
zh-tw       1
mk          1
tr          1
lt          1
fa          1
Name: count, dtype: int64
Jumlah review bahasa Inggris: 1068


Setelah difilter, ada beberpa bahasa lain seperti **Wales**, **Indonesian**, **Somali**, dan **Vietnamese** semuanya saya filter suapaya **English only**, sesuai perintah soal, hasilnya tersisa sekitar **1068** review.

#### Simpan dataset akhir ke CSV

In [10]:
df_en = df_en.drop(columns=["lang"])

Buang kolom **lang** karena sudah tidak terpakai dan sudah **100% english** sehingga kolom tersebut tidak lagi memberikan **informasi tambahan** untuk analisis berikutnya.

In [11]:
# pastikan kolom tanggal sudah datetime
df_en['review_date'] = pd.to_datetime(df_en['review_date'])

# urutkan dari review terbaru ke lama
df_en = df_en.sort_values('review_date', ascending=False).reset_index(drop=True)

# ambil hanya 1000 review teratas (paling baru) dan buang sisanya
df_en = df_en.iloc[:1000].reset_index(drop=True)

print(len(df_en))

1000


Disini aku membuang dataset 68 berlebih tadi biar 100% sesuai dengan permintaan soal, aku urutkan dari yang terbaru, baru dibuang 68 sisanya

In [12]:
df_en.to_csv("hsr_reviews_en.csv", index=False)

Simpan dataset untuk dianalisa lebih lanjut